# Generate sparse matrices in the spirit of Sparse Attention (BigBird)

In [74]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.sparse import coo_matrix
from scipy.io import mmwrite

block_size = 64
w = 1 # window blocks
r = 8 # random blocks
g = 2 # global blocks

n = 65536

def generate_global_blocks(n, block_size, g):
    Ln = list(range(n))
    Lb = list(range(block_size * g))

    J = Ln * block_size * g + (n - g * block_size) * Lb
    I = []
    for i in range(g * block_size):
        I.extend([i] * n)
    for i in range(g * block_size, n):
        I.extend([i] * (g * block_size))
    return I, J

def generate_window_blocks(n, block_size, w):
    I = []
    J = []
    # diagonal blocks
    for i in range(0, n, block_size):
        Lc = list(range(i, i + block_size))
        J.extend(Lc * block_size)
        for j in range(block_size):
            I.extend([i+j] * block_size)
    # upper diagonal blocks
    for d in range(1, w):
        for i in range(0, n - d * block_size, block_size):
            Lc = list(range(d * block_size + i, (d+1) * block_size + i))
            J.extend(Lc * block_size)
            for j in range(block_size):
                I.extend([i+j] * block_size)
    # lower diagonal blocks
    for d in range(1, w):
        for i in range(d * block_size, n, block_size):
            Lc = list(range(i-d * block_size, i - (d-1) * block_size))
            J.extend(Lc * block_size)
            for j in range(block_size):
                I.extend([i+j] * block_size)
    return I, J

def generate_random_blocks(n, block_size, r, g):
    
    J = []
    I = []
    total_blocks = n // block_size
    for i in range(g, total_blocks):
        rand_indices = np.random.permutation(total_blocks)[0:r]
        for j in range(r):
            for b in range(block_size):
                I.extend([i * block_size + b] * block_size)
            j_index = rand_indices[j]
            Lc = list(range(j_index * block_size, (j_index + 1) * block_size))
            J.extend(Lc * block_size)
    return I, J 

print("generating global blocks")
row_g, col_g = generate_global_blocks(n, block_size, g)
print(len(row_g))
print(len(col_g))

print("generating window blocks")
row_w, col_w = generate_window_blocks(n, block_size, w)
print(len(row_w))
print(len(col_w))

print("generating random blocks")
row_r, col_r = generate_random_blocks(n, block_size, r, g)
print(len(row_r))
print(len(col_r))

row = row_w + row_g
col = col_w + col_g

row = row_w + row_g + row_r
col = col_w + col_g + col_r
val = np.random.randint(0,8,len(col))
A = coo_matrix((val, (row,col)), (n,n))
mmwrite(f'./b-{block_size}_r-{r}_w-{w}_g-{g}_n-{n}.mtx', A)
print(A.shape)
print(n*n)
print(A.nnz)
# plt.spy(A, markersize=0.5)
# plt.show()

generating global blocks
16760832
16760832
generating window blocks
4194304
4194304
generating random blocks
33488896
33488896
(65536, 65536)
4294967296
54444032
